In [ ]:
# Cell 1: Setup
import asyncio
from bufferiq.core.config import get_settings
from bufferiq.core.database import DatabaseManager
from bufferiq.ml.analysis import (
    DataLoader,
    EngagementAnalyzer,
    TemporalAnalyzer,
    ContentAnalyzer,
    Visualizer
)

settings = get_settings()

In [ ]:
# Cell 2: Load Data
async def load_data():
    db_manager = DatabaseManager(settings)
    await db_manager.connect()
    
    async with db_manager.session() as session:
        loader = DataLoader(session)
        df = await loader.load_posts(status="sent")
    
    await db_manager.disconnect()
    return df

df = await load_data()
print(f"Loaded {len(df)} posts")
df.head()


In [ ]:
# Cell 3: Engagement Analysis
analyzer = EngagementAnalyzer()
df = analyzer.calculate_engagement_rate(df)
stats = analyzer.analyze_distribution(df, "engagement_rate")
print(stats)

In [ ]:
# Cell 4: Temporal Analysis
temporal = TemporalAnalyzer()
hourly = temporal.hourly_patterns(df)
hourly.sort_values("mean", ascending=False).head(10)

In [ ]:
# Cell 5: Visualizations
viz = Visualizer("../outputs/figures")
viz.plot_distribution(df['engagement_rate'], "Engagement Distribution")
viz.plot_hourly_heatmap(df)
viz.plot_platform_comparison(df, 'engagement_rate')